In [0]:
spark.conf.set("fs.azure.account.auth.type.nyctaxiprojectdatalake.dfs.core.windows.net", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.nyctaxiprojectdatalake.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.nyctaxiprojectdatalake.dfs.core.windows.net", Application_ID)
spark.conf.set("fs.azure.account.oauth2.client.secret.nyctaxiprojectdatalake.dfs.core.windows.net", Secret_ID)
spark.conf.set("fs.azure.account.oauth2.client.endpoint.nyctaxiprojectdatalake.dfs.core.windows.net", "https://login.microsoftonline.com/Directory_ID")

In [0]:
dbutils.fs.ls("abfss://bronzelayer@nyctaxiprojectdatalake.dfs.core.windows.net/")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df_trip_type = spark.read.format("csv")\
                      .option("inferSchema", True)\
                      .option("header", True)\
                      .load("abfss://bronzelayer@nyctaxiprojectdatalake.dfs.core.windows.net/trip_type")

In [0]:
df_trip_type.display()

In [0]:
df_trip_zone = spark.read.format("csv")\
                         .options(inferSchema=True, header=True)\
                         .load("abfss://bronzelayer@nyctaxiprojectdatalake.dfs.core.windows.net/trip_zone")

In [0]:
df_trip_zone.display()

In [0]:
taxi_data_schema = '''
                VendorID BIGINT,
                lpep_pickup_datetime TIMESTAMP,
                lpep_dropoff_datetime TIMESTAMP,
                store_and_fwd_flag STRING,
                RatecodeID BIGINT,
                PULocationID BIGINT,
                DOLocationID BIGINT,
                passenger_count BIGINT,
                trip_distance DOUBLE,
                fare_amount DOUBLE,
                extra DOUBLE,
                mta_tax DOUBLE,
                tip_amount DOUBLE,
                tolls_amount DOUBLE,
                ehail_fee DOUBLE,
                improvement_surcharge DOUBLE,
                total_amount DOUBLE,
                payment_type BIGINT,
                trip_type BIGINT,
                congestion_surcharge DOUBLE

      '''

In [0]:
df_taxi_data = spark.read.format("parquet")\
                         .schema(taxi_data_schema)\
                         .option("header", True)\
                         .option("recursiveFileLookup", True)\
                         .load("abfss://bronzelayer@nyctaxiprojectdatalake.dfs.core.windows.net/Taxi2025Data")

In [0]:
df_taxi_data.display()

In [0]:
df_trip_type = df_trip_type.withColumnRenamed('description', 'trip_description')
df_trip_type.display()

In [0]:
df_trip_type.write.format('parquet')\
                        .mode('append')\
                        .option('path', 'abfss://silverlayer@nyctaxiprojectdatalake.dfs.core.windows.net/trip_type')\
                        .save()

In [0]:
df_trip_zone = df_trip_zone.withColumn('Zone1', split(col('Zone'), '/')[0])\
                           .withColumn('Zone2', split(col('Zone'), '/')[1])
df_trip_zone.display()

In [0]:
df_trip_zone.write.format('parquet')\
                  .mode('append')\
                  .option('path', 'abfss://silverlayer@nyctaxiprojectdatalake.dfs.core.windows.net/trip_zone')\
                  .save()

In [0]:
df_taxi_data = df_taxi_data.withColumn('trip_date', to_date('lpep_pickup_datetime'))\
                          .withColumn('trip_month', month('lpep_pickup_datetime'))\
                          .withColumn('trip_year', year('lpep_pickup_datetime'))

df_taxi_data = df_taxi_data.select('VendorID', 'PULocationID', 'DOLocationID', 'fare_amount', 'total_amount','trip_date', 'trip_month', 'trip_year')

df_taxi_data.display()

In [0]:
df_taxi_data.write.format('parquet').mode('append').option('path', 'abfss://silverlayer@nyctaxiprojectdatalake.dfs.core.windows.net/trip_data').save()

In [0]:
df_taxi_data.display()

Databricks visualization. Run in Databricks to view.